# Silver Pedidos

**Fuente:** `brz_dataco.sharepoint.raw_supply_chain` (histórico completo acumulado)

**Destino:** `slv_dataco.supply_chain.hd_pedidos`

**Grano:** un registro por `order_id`

## Qué hace

1. Lee todo el histórico de Bronze acumulado, no solo el último corte.
2. Separa los registros con `order_id` nulo, vacío o `'0'`: no tienen clave para deduplicar, así que se conservan y se marcan en cuarentena.
3. Deduplica el resto por `order_id`, priorizando el registro del corte más reciente (`dedup_reciente`, `fecha_fin_corte` descendente).
4. Tipa los datos, que en Bronze están como texto:

   * `order_date_dateorders` y `shipping_date_dateorders` se convierten a fecha y hora con `try_to_timestamp` y los formatos definidos en el notebook. Si una fecha no calza con ningún formato, queda en `NULL` en lugar de detener el notebook.
   * `order_customer_id`, `late_delivery_risk`, `days_for_shipping_real` y `days_for_shipment_scheduled`: `int`.
   * Campos de texto: `trim` + mayúsculas.
5. Marca la calidad de cada registro (ver más abajo).
6. Conserva las columnas de trazabilidad provenientes de Bronze.
7. Escribe la tabla Silver mediante `overwrite` completo.
8. Registra en el esquema de control cuántas filas se deduplicaron y cuántas quedaron en cuarentena.

## Columnas de negocio

* `order_id`
* `order_customer_id`
* `order_date`
* `shipping_date`
* `order_status`
* `delivery_status`
* `late_delivery_risk`
* `days_for_shipping_real`
* `days_for_shipment_scheduled`
* `shipping_mode`
* `market`
* `order_region`
* `order_country`
* `order_state`
* `order_city`
* `order_zipcode`

## Calidad y cuarentena

Ningún registro se elimina. Los que incumplen una regla se conservan marcados y Gold no los lee.

| Columna | Valores |
|---|---|
| `estado_calidad` | `VALIDO` o `CUARENTENA` |
| `motivo_cuarentena` | `ORDER_ID_INVALIDO`, `FECHA_PEDIDO_INVALIDA`, `CLIENTE_NO_INFORMADO`, `RIESGO_RETRASO_INVALIDO` o `NULL` (un solo motivo por registro: el primero que se cumpla) |
| `fecha_procesamiento` | Momento en que Silver procesó el registro |

## Trazabilidad conservada

* `fecha_inicio_corte`
* `fecha_fin_corte`
* `fecha_carga`
* `id_ejecucion`

## Auditoría de la ejecución

Al terminar, el notebook deja evidencia en `metadata_dataco.supply_chain`:

* `hd_deduplicacion_log`: una fila por ejecución. Registra los pedidos únicos por `order_id`, las filas agrupadas (alto y esperable, porque cada pedido trae varios ítems) y los duplicados reales por `order_id` + `order_item_id`, que deben ser **0**.
* `hd_cuarentena_resumen`: filas aisladas por regla en esa ejecución, incluso cuando son 0.

## Por qué importa este notebook

La tabla de **Pedidos** representa la cabecera de los pedidos y es una de las principales fuentes para los indicadores de Gold.

Los filtros de negocio específicos de cada indicador, incluida la exclusión de pedidos cancelados, se aplican en Gold y no en esta tabla Silver. La exclusión por calidad (`estado_calidad`) sí se aplica en Gold, que solo lee registros `VALIDO`.

In [0]:
# ============================================================
# Celda 1: Parámetros + lectura de Bronze + helpers
# ------------------------------------------------------------
# Mismo patrón que las 3 dimensiones. Notebook con sesión propia,
# por eso repite todo desde cero.
# ============================================================

dbutils.widgets.text("bronze_catalog", "brz_dataco", "Catálogo Bronze")
dbutils.widgets.text("bronze_schema", "sharepoint", "Esquema Bronze")
dbutils.widgets.text("bronze_table", "raw_supply_chain", "Tabla Bronze")

dbutils.widgets.text("silver_catalog", "slv_dataco", "Catálogo Silver")
dbutils.widgets.text("silver_schema", "supply_chain", "Esquema Silver")

bronze_catalog = dbutils.widgets.get("bronze_catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
bronze_table = dbutils.widgets.get("bronze_table")

silver_catalog = dbutils.widgets.get("silver_catalog")
silver_schema = dbutils.widgets.get("silver_schema")

BRONZE_TABLE_FQN = f"{bronze_catalog}.{bronze_schema}.{bronze_table}"
SILVER_PEDIDOS_FQN = f"{silver_catalog}.{silver_schema}.hd_pedidos"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{silver_catalog}.{silver_schema}"
)

df_bronze = spark.table(BRONZE_TABLE_FQN)

from pyspark.sql import functions as F, Window

def clean_str(col_name: str, default: str = "") -> F.Column:
    return F.upper(F.trim(F.coalesce(F.col(col_name).cast("string"), F.lit(default))))

def clean_num(col_name: str, cast_type) -> F.Column:
    # Vacío o texto no convertible -> NULL (nunca 0).
    texto = f"NULLIF(TRIM(CAST(`{col_name}` AS STRING)), '')"
    return F.expr(f"try_cast({texto} AS {cast_type})")

def tipo_invalido(col_name: str, cast_type) -> F.Column:
    # Verdadero si la celda trae texto (no vacío) que no se puede convertir al tipo.
    texto = f"NULLIF(TRIM(CAST(`{col_name}` AS STRING)), '')"
    return F.expr(f"{texto} IS NOT NULL AND try_cast({texto} AS {cast_type}) IS NULL")

def dedup_reciente(df, partition_cols: list, orden_col: str):
    w = Window.partitionBy(*partition_cols).orderBy(F.col(orden_col).desc())
    return (
        df.withColumn("rn", F.row_number().over(w))
          .filter(F.col("rn") == 1)
          .drop("rn")
    )

print(f"Total filas en Bronze: {df_bronze.count()}")
print("✓ Helpers definidos: clean_str, clean_num, dedup_reciente")

In [0]:
# ============================================================
# INICIO DE LOG DE EJECUCIÓN — slv_orders
# ============================================================

import uuid

dbutils.widgets.text("metadata_catalog", "metadata_dataco", "Catálogo Metadata")
dbutils.widgets.text("metadata_schema", "supply_chain", "Esquema Metadata")
dbutils.widgets.text("id_ejecucion_manual", "", "Id de ejecución (solo corridas manuales)")

metadata_catalog = dbutils.widgets.get("metadata_catalog")
metadata_schema = dbutils.widgets.get("metadata_schema")

TABLA_ETL_LOG = f"{metadata_catalog}.{metadata_schema}.hd_etl_log"
TABLA_DEDUP_LOG = f"{metadata_catalog}.{metadata_schema}.hd_deduplicacion_log"
TABLA_CUARENTENA_RESUMEN = f"{metadata_catalog}.{metadata_schema}.hd_cuarentena_resumen"

id_manual = dbutils.widgets.get("id_ejecucion_manual")
try:
    id_ejecucion = dbutils.jobs.taskValues.get(taskKey="prepare_pipeline_control", key="id_ejecucion")
except Exception:
    if not id_manual:
        raise
    id_ejecucion = id_manual

LOG_ID = str(uuid.uuid4())
NOTEBOOK = "slv_orders"
CAPA = "silver"

spark.sql(f"""
INSERT INTO {TABLA_ETL_LOG}
(log_id, id_ejecucion, notebook, capa, estado, fecha_inicio)
VALUES ('{LOG_ID}', '{id_ejecucion}', '{NOTEBOOK}', '{CAPA}', 'EN_PROCESO', current_timestamp())
""")

try:
    dbutils.jobs.taskValues.set(key="log_id", value=LOG_ID)
    print("✓ log_id publicado como Task Value")
except Exception:
    if not id_manual:
        raise
    print("• Corrida manual: log_id no se publica como Task Value")

print(f"✓ Registro EN_PROCESO creado en etl_log: {LOG_ID}")

In [0]:
[c for c in df_bronze.columns if any(k in c.lower() for k in ["order_id", "order_customer_id", "order_status", "delivery_status", "days_for_ship", "shipping_mode", "date", "late_delivery"])]

In [0]:
df_bronze.select("order_date_dateorders", "fecha_pedido_carga", "shipping_date_dateorders").printSchema()
df_bronze.select("order_date_dateorders", "fecha_pedido_carga", "shipping_date_dateorders").show(5, truncate=False)

In [0]:
# p ============================================================
# Celda 2: Parseo de fechas + Deduplicación + Cuarentena + construcción de hd_pedidos
# ------------------------------------------------------------
# Fechas: uso order_date_dateorders / shipping_date_dateorders (string,
# formato "M/d/yyyy H:mm") parseadas con try_to_timestamp: si el valor no
# calza con ningún formato devuelve NULL en vez de fallar (ANSI).
#
# Dedup: por order_id, determinístico (fecha_fin_corte desc).
#
# Cuarentena (no se elimina nada):
#   - order_id nulo/vacío/'0' -> ORDER_ID_INVALIDO. Sin clave no hay con qué
#     deduplicar, así que estas filas se conservan tal cual.
#   - fecha de pedido no parseable -> FECHA_PEDIDO_INVALIDA.
#   - cliente nulo, vacío o 0 -> CLIENTE_NO_INFORMADO.
#   - late_delivery_risk distinto de 0 y 1 (o vacío) -> RIESGO_RETRASO_INVALIDO.
#   - días de envío con texto no numérico -> TIPO_DATO_INVALIDO.
#   Cada fila queda con un solo motivo: el primero que se cumpla, en ese orden.
# Gold solo lee estado_calidad = 'VALIDO'.
# ============================================================

FECHA_FORMATOS = ["M/d/yyyy H:mm", "MM/dd/yyyy HH:mm:ss", "MM/dd/yyyy"]

def clean_date(col_name: str) -> F.Column:
    resultado = F.lit(None).cast("timestamp")
    for fmt in FECHA_FORMATOS:
        resultado = F.coalesce(
            resultado,
            F.expr(f"try_to_timestamp(trim(CAST(`{col_name}` AS STRING)), '{fmt}')")
        )
    return resultado

order_id_numero = clean_num("order_id", "int")
id_invalido = order_id_numero.isNull() | (order_id_numero == 0)

df_id_invalido = df_bronze.filter(id_invalido)
df_id_valido = df_bronze.filter(~id_invalido)

df_pedidos_dedup = dedup_reciente(df_id_valido, ["order_id"], orden_col="fecha_fin_corte")
df_pedidos_base = df_pedidos_dedup.unionByName(df_id_invalido)

fecha_pedido = clean_date("order_date_dateorders")

cliente_id = clean_num("order_customer_id", "int")
tipo_dato_invalido = (tipo_invalido("days_for_shipping_real", "int")
                      | tipo_invalido("days_for_shipment_scheduled", "int"))
riesgo_valido = F.coalesce(
    F.trim(F.col("late_delivery_risk").cast("string")).isin("0", "1"),
    F.lit(False)
)
motivo = (
    F.when(id_invalido, F.lit("ORDER_ID_INVALIDO"))
     .when(fecha_pedido.isNull(), F.lit("FECHA_PEDIDO_INVALIDA"))
     .when(cliente_id.isNull() | (cliente_id == 0), F.lit("CLIENTE_NO_INFORMADO"))
     .when(~riesgo_valido, F.lit("RIESGO_RETRASO_INVALIDO"))
     .when(tipo_dato_invalido, F.lit("TIPO_DATO_INVALIDO"))
)

df_silver_pedidos = df_pedidos_base.select(
    clean_num("order_id", "int").alias("order_id"),
    cliente_id.alias("order_customer_id"),
    fecha_pedido.alias("order_date"),
    clean_date("shipping_date_dateorders").alias("shipping_date"),
    clean_str("order_status").alias("order_status"),
    clean_str("delivery_status").alias("delivery_status"),
    clean_num("late_delivery_risk", "int").alias("late_delivery_risk"),
    clean_num("days_for_shipping_real", "int").alias("days_for_shipping_real"),
    clean_num("days_for_shipment_scheduled", "int").alias("days_for_shipment_scheduled"),
    clean_str("shipping_mode").alias("shipping_mode"),
    clean_str("market").alias("market"),
    clean_str("order_region").alias("order_region"),
    clean_str("order_country").alias("order_country"),
    clean_str("order_state").alias("order_state"),
    clean_str("order_city").alias("order_city"),
    clean_str("order_zipcode").alias("order_zipcode"),

    # Calidad
    motivo.alias("motivo_cuarentena"),
    F.when(motivo.isNotNull(), F.lit("CUARENTENA"))
    .otherwise(F.lit("VALIDO"))
    .alias("estado_calidad"),
    F.current_timestamp().alias("fecha_procesamiento"),

    # Trazabilidad
    F.col("fecha_inicio_corte"),
    F.col("fecha_fin_corte"),
    F.col("fecha_carga"),
    F.col("id_ejecucion")
)

print(f"Filas en Bronze                 : {df_bronze.count()}")
print(f"Filas en hd_pedidos             : {df_silver_pedidos.count()}")
print(f"  de las cuales en cuarentena   : {df_silver_pedidos.filter(F.col('estado_calidad') == 'CUARENTENA').count()}")

display(df_silver_pedidos.limit(10))

In [0]:
# ============================================================
# Celda 3: Escritura a silver_pedidos
# ------------------------------------------------------------
# overwrite completo, mismo patrón que las 3 dimensiones.
# ============================================================

(
    df_silver_pedidos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_PEDIDOS_FQN)
)

print(f"✓ {df_silver_pedidos.count()} filas escritas en {SILVER_PEDIDOS_FQN}")

In [0]:
# ============================================================
# Registro de deduplicación y cuarentena de esta ejecución
# ------------------------------------------------------------
# Una fila por ejecución y tabla Silver:
#   filas_agrupadas  -> ítems de Bronze agrupados en una fila por pedido (informativa).
#   duplicados_reales -> filas repetidas de (order_id, order_item_id) en Bronze; debe ser 0.
# Las filas con order_id inválido no entran aquí: van a cuarentena.
# El DELETE previo evita duplicar el registro si la task se reintenta.
# El criterio de sobreviviente (fecha_fin_corte DESC) queda en el markdown del notebook.
# ============================================================
TABLA_SILVER = "hd_pedidos"

filas_entrada = df_id_valido.count()
filas_salida = df_pedidos_dedup.count()
filas_unicas_bronze = df_id_valido.select("order_id", "order_item_id").distinct().count()
duplicados_reales = filas_entrada - filas_unicas_bronze

spark.sql(f"DELETE FROM {TABLA_DEDUP_LOG} WHERE id_ejecucion = '{id_ejecucion}' AND tabla_silver = '{TABLA_SILVER}'")
spark.sql(f"""
INSERT INTO {TABLA_DEDUP_LOG}
SELECT '{id_ejecucion}', '{TABLA_SILVER}', 'order_id',
       {filas_entrada}, {filas_salida}, {filas_entrada - filas_salida},
       'order_id, order_item_id', {duplicados_reales}, current_timestamp()
""")

spark.sql(f"DELETE FROM {TABLA_CUARENTENA_RESUMEN} WHERE id_ejecucion = '{id_ejecucion}' AND tabla_silver = '{TABLA_SILVER}'")
spark.sql(f"""
INSERT INTO {TABLA_CUARENTENA_RESUMEN}
SELECT '{id_ejecucion}', '{TABLA_SILVER}', r.motivo, COALESCE(c.filas, 0), current_timestamp()
FROM (SELECT explode(array('ORDER_ID_INVALIDO', 'FECHA_PEDIDO_INVALIDA',
                           'CLIENTE_NO_INFORMADO', 'RIESGO_RETRASO_INVALIDO','TIPO_DATO_INVALIDO')) AS motivo) r
LEFT JOIN (
    SELECT motivo_cuarentena, COUNT(*) AS filas
    FROM {SILVER_PEDIDOS_FQN}
    WHERE estado_calidad = 'CUARENTENA'
    GROUP BY motivo_cuarentena
) c ON c.motivo_cuarentena = r.motivo
""")

print(f"✓ {filas_entrada} filas de Bronze -> {filas_salida} pedidos | agrupadas: {filas_entrada - filas_salida} | duplicados reales: {duplicados_reales}")
display(spark.table(TABLA_DEDUP_LOG).filter(f"id_ejecucion = '{id_ejecucion}'"))

In [0]:
TABLA_DESTINO = SILVER_PEDIDOS_FQN   # cambia por notebook (ver tabla)
filas_procesadas = spark.table(TABLA_DESTINO).count()

spark.sql(f"""
UPDATE {TABLA_ETL_LOG}
SET estado = 'EXITOSO',
    fecha_fin = current_timestamp(),
    filas_procesadas = {filas_procesadas}
WHERE log_id = '{LOG_ID}'
""")

print(f"✓ etl_log actualizado: EXITOSO ({filas_procesadas} filas en {TABLA_DESTINO})")